In [38]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from dotenv import load_dotenv, find_dotenv
import os
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_text_splitters import CharacterTextSplitter

_ = load_dotenv(find_dotenv())

In [39]:
# Define paths
docs_path = "docs"
persistent_directory = "dbsmall4/chroma_db"
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

In [40]:
def load_documents(docs_path="docs"):
    """Load all text files from the docs directory"""
    print(f"Loading documents from {docs_path}...")
    
    # Check if docs directory exists
    if not os.path.exists(docs_path):
        raise FileNotFoundError(f"The directory {docs_path} does not exist. Please create it and add your company files.")
    
    # Load all .txt files from the docs directory
    loader = DirectoryLoader(
        path=docs_path,
        glob="*.txt",
        loader_cls=TextLoader
    )
    documents = loader.load()
    if len(documents) == 0:
        raise FileNotFoundError(f"No .txt files found in {docs_path}. Please add your company documents.")
    
    return documents


In [73]:
# delete the vector store directory if it exists
if os.path.exists(persistent_directory):
    vectorstore = Chroma(
            persist_directory=persistent_directory,
            embedding_function=embedding_model, 
            collection_metadata={"hnsw:space": "cosine"}
        )
    vectorstore.reset_collection()
    print(f"Cleaned up the existing vector store at {persistent_directory}")
    


Cleaned up the existing vector store at dbsmall4/chroma_db


In [56]:
# Step 1: Load documents
documents = load_documents(docs_path)    # just load one document for now
print(type(documents))  # should be a list of Document objects
print(type(documents[0]))
for i, doc in enumerate(documents):
        print(f"\nDocument {i+1}:")
        print(f"  Source: {doc.metadata['source']}")
        print(f"  Content length: {len(doc.page_content)} characters")
        # print(f"  Content preview: {doc.page_content[:100]}...")
        print(f"  metadata: {doc.metadata}")

Loading documents from docs...
<class 'list'>
<class 'langchain_core.documents.base.Document'>

Document 1:
  Source: docs/Tesla.txt
  Content length: 299196 characters
  metadata: {'source': 'docs/Tesla.txt'}

Document 2:
  Source: docs/Microsoft.txt
  Content length: 201014 characters
  metadata: {'source': 'docs/Microsoft.txt'}

Document 3:
  Source: docs/Nvidia.txt
  Content length: 148417 characters
  metadata: {'source': 'docs/Nvidia.txt'}

Document 4:
  Source: docs/SpaceX.txt
  Content length: 206132 characters
  metadata: {'source': 'docs/SpaceX.txt'}

Document 5:
  Source: docs/Google.txt
  Content length: 232201 characters
  metadata: {'source': 'docs/Google.txt'}


In [43]:
def split_documents(documents, chunk_size=1000, chunk_overlap=0):
    """Split documents into smaller chunks with overlap"""
    print("Splitting documents into chunks...")
    
    text_splitter = CharacterTextSplitter(
        chunk_size=chunk_size,  
        chunk_overlap=chunk_overlap
    )
    
    chunks = text_splitter.split_documents(documents)
    return chunks

In [53]:
# create a unique chunk ID using document name and hash of chunk text
import hashlib

def chunk_id(document_name: str, chunk_text: str) -> str:
    h = hashlib.sha256(chunk_text.encode("utf-8")).hexdigest()
    return f"{document_name}:{h}"

In [ ]:
# Step 2: Split into chunks
chunks = split_documents(documents)
print(f"Number of chunnks at start:{len(chunks)}")
print(type(chunks[0]))

for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i+1} ---")
    print(f"Source: {chunk.metadata['source']}")
    print(f"Length: {len(chunk.page_content)} characters")
    # add the document name from metadata
    document_name = chunk.metadata['source'].split("/")[-1]
    print(f"Document Name: {document_name}")
    # add the document name to metadata
    chunk.metadata['document_name'] = document_name
    chunk.metadata['chunk_index'] = i
    print(f"Metadata: {chunk.metadata}")
    chunk.id = chunk_id(document_name, chunk.page_content)

for i, chunk in enumerate(chunks[:2]):
    print(f"\n--- Chunk {i+1} ---")
    print(f"Metadata: {chunk.metadata}")
    print(f"Chunk ID: {chunk.id}")

print(f"Number of chunks at end:{len(chunks)}")

Created a chunk of size 1029, which is longer than the specified 1000
Created a chunk of size 1206, which is longer than the specified 1000
Created a chunk of size 1055, which is longer than the specified 1000
Created a chunk of size 1082, which is longer than the specified 1000
Created a chunk of size 1027, which is longer than the specified 1000
Created a chunk of size 1187, which is longer than the specified 1000
Created a chunk of size 1518, which is longer than the specified 1000
Created a chunk of size 1603, which is longer than the specified 1000
Created a chunk of size 1436, which is longer than the specified 1000
Created a chunk of size 1039, which is longer than the specified 1000
Created a chunk of size 1078, which is longer than the specified 1000
Created a chunk of size 1043, which is longer than the specified 1000
Created a chunk of size 1019, which is longer than the specified 1000
Created a chunk of size 1068, which is longer than the specified 1000
Created a chunk of s

Splitting documents into chunks...
Number of chunnks at start:1330
<class 'langchain_core.documents.base.Document'>

--- Chunk 1 ---
Source: docs/Tesla.txt
Length: 969 characters
Document Name: Tesla.txt
Metadata: {'source': 'docs/Tesla.txt', 'document_name': 'Tesla.txt', 'chunk_index': 0}

--- Chunk 2 ---
Source: docs/Tesla.txt
Length: 666 characters
Document Name: Tesla.txt
Metadata: {'source': 'docs/Tesla.txt', 'document_name': 'Tesla.txt', 'chunk_index': 1}

--- Chunk 3 ---
Source: docs/Tesla.txt
Length: 1000 characters
Document Name: Tesla.txt
Metadata: {'source': 'docs/Tesla.txt', 'document_name': 'Tesla.txt', 'chunk_index': 2}

--- Chunk 4 ---
Source: docs/Tesla.txt
Length: 788 characters
Document Name: Tesla.txt
Metadata: {'source': 'docs/Tesla.txt', 'document_name': 'Tesla.txt', 'chunk_index': 3}

--- Chunk 5 ---
Source: docs/Tesla.txt
Length: 921 characters
Document Name: Tesla.txt
Metadata: {'source': 'docs/Tesla.txt', 'document_name': 'Tesla.txt', 'chunk_index': 4}

--- Chu

In [58]:
def create_vector_store(chunks, persist_directory):
    """Create and persist ChromaDB vector store"""
    print("Creating embeddings and storing in ChromaDB...")
            
    # Create ChromaDB vector store
    print("--- Creating vector store ---")
    vectorstore = Chroma.from_documents(
        collection_name="documents",
        documents=chunks,
        embedding=embedding_model,
        persist_directory=persist_directory, 
        collection_metadata={"hnsw:space": "cosine"}
    )
    print("--- Finished creating vector store ---")
    print(f"Vector store created and saved to {persist_directory}")
    return vectorstore

In [95]:
# Step 3: Create vector store
db = create_vector_store(chunks, persistent_directory)
print("\n✅ Ingestion complete! Your documents are now ready for RAG queries.")


Creating embeddings and storing in ChromaDB...
--- Creating vector store ---
--- Finished creating vector store ---
Vector store created and saved to dbsmall4/chroma_db

✅ Ingestion complete! Your documents are now ready for RAG queries.


In [97]:
# get the collection name
# db.reset_collection()  # reset collection to avoid duplicates during repeated runs
collection_name = db._collection.name
print(f"Collection name: {collection_name}")
count = db._collection.count()
print(f"Number of vectors: {count}")


Collection name: documents
Number of vectors: 1330


In [98]:
docs = []
for chunk in chunks:
    docs.append(
        Document(
            page_content=chunk.page_content,
            metadata=chunk.metadata,
            id=chunk_id(chunk.metadata["document_name"], chunk.page_content)
        )
    )
db.add_documents(docs)
count = db._collection.count()
print(f"Number of vectors: {count}")

Number of vectors: 1330


In [85]:
print(docs[0].id)


Tesla.txt:bab17d0126b19137800b1036d75fd7465452a68a76250ac2ac753337d411641f


In [48]:
# Retrieval without using document name filter

query = "Who is Larry Elison?" 

retriever = db.as_retriever(search_kwargs={"k": 5})
# retriever = db.as_retriever(
#     search_type="similarity_score_threshold",
#     search_kwargs={
#         "k": 5,
#         "score_threshold": 0.3  # Only return chunks with cosine similarity ≥ 0.3
#     }
# )
relevant_docs = retriever.invoke(query)
for i, chunk in enumerate(relevant_docs):
    print(f"\n--- Chunk {i+1} ---")
    print(f"Metadata: {chunk.metadata}")
    print(f"Content: {chunk.page_content[0:100]}")



--- Chunk 1 ---
Metadata: {'chunk_index': 360, 'source': 'docs/Tesla.txt', 'document_name': 'Tesla.txt'}
Content: 578. Waters, Richard (April 12, 2017). "Tesla investors seek stronger boardroom controls" (https://w

--- Chunk 2 ---
Metadata: {'source': 'docs/Tesla.txt', 'chunk_index': 289, 'document_name': 'Tesla.txt'}
Content: 351. Powell, Jamie; Jones, Claire (December 18, 2019). "The question of Tesla's cash to be collected

--- Chunk 3 ---
Metadata: {'chunk_index': 361, 'source': 'docs/Tesla.txt', 'document_name': 'Tesla.txt'}
Content: 581. Ponciano, Jonathan (June 10, 2022). "Tesla Files For Another Stock Split—Reveals Billionaire La

--- Chunk 4 ---
Metadata: {'chunk_index': 364, 'source': 'docs/Tesla.txt', 'document_name': 'Tesla.txt'}
Content: 589. Kolodny, Lora (November 8, 2018). "Robyn Denholm replaces Elon Musk as Tesla's board chair" (ht

--- Chunk 5 ---
Metadata: {'source': 'docs/Tesla.txt', 'chunk_index': 142, 'document_name': 'Tesla.txt'}
Content: 46. Hamilton, Isobel 

## Query Within a Specific Document (Level-1 Filter)

In [51]:
# force the retriver to look into a specific document using metadata filter
retriever1 = db.as_retriever(
    search_kwargs={
        "k": 5,
        "filter": {"document_name": "Nvidia.txt"}
    }
)

relevant_docs1 = retriever1.invoke(query)
for i, chunk in enumerate(relevant_docs1):
    print(f"\n--- Chunk {i+1} ---")
    print(f"Metadata: {chunk.metadata}")
    print(f"Content: {chunk.page_content[0:100]}")




--- Chunk 1 ---
Metadata: {'document_name': 'Nvidia.txt', 'source': 'docs/Nvidia.txt', 'chunk_index': 652}
Content: Rob Burgess (former chief executive officer of Macromedia Inc.)
Tench Coxe (former managing director

--- Chunk 2 ---
Metadata: {'chunk_index': 617, 'source': 'docs/Nvidia.txt', 'document_name': 'Nvidia.txt'}
Content: Microsystems; and Curtis Priem, who was previously References:[1][2]
a senior staff engineer and gra

--- Chunk 3 ---
Metadata: {'source': 'docs/Nvidia.txt', 'chunk_index': 728, 'document_name': 'Nvidia.txt'}
Content: 121. Emilia David (September 25, 2023). "Getty made an AI generator that only trained on its
license

--- Chunk 4 ---
Metadata: {'chunk_index': 619, 'document_name': 'Nvidia.txt', 'source': 'docs/Nvidia.txt'}
Content: Huang.[30] Priem broke that deadlock by resigning first from
Sun, effective December 31, 1992.[30] A

--- Chunk 5 ---
Metadata: {'document_name': 'Nvidia.txt', 'source': 'docs/Nvidia.txt', 'chunk_index': 687}
Content: 33. William